# Первичный анализ протокола ASVspoof 5

Notebook исследует только официальный файл протокола и не скачивает аудиодатасет. Цель — увидеть баланс REAL/FAKE, атаки и доступные условия записи до выбора подмножества для первого обучения CNN.

In [ ]:
# 1. Проверка среды выполнения Python
import platform
import sys

print('Python:', sys.version)
print('Платформа:', platform.platform())
print('Исполняемый файл:', sys.executable)

In [ ]:
# 2. Импорт библиотек
from pathlib import Path
from urllib.request import urlretrieve

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', None)

## Получение протокола

Официальный репозиторий baseline ASVspoof 5 описывает формат, но не содержит публичной прямой ссылки на архив протоколов. Поэтому URL ниже намеренно не заполнен. Укажите подтверждённую официальную ссылку или загрузите полученный официальный файл `ASVspoof5.train.tsv` в Colab вручную. Аудиофайлы для этого notebook не нужны.

In [ ]:
# 3. Получение официального протокола без полного аудиодатасета
PROTOCOL_URL = ''  # Вставьте только проверенную официальную ссылку
PROTOCOL_PATH = Path('/content/ASVspoof5.train.tsv')

if PROTOCOL_URL:
    urlretrieve(PROTOCOL_URL, PROTOCOL_PATH)
    print('Протокол загружен:', PROTOCOL_PATH)
elif not PROTOCOL_PATH.exists():
    try:
        from google.colab import files
        print('Выберите официальный файл ASVspoof5.train.tsv')
        uploaded = files.upload()
        uploaded_name = next(iter(uploaded))
        PROTOCOL_PATH = Path('/content') / uploaded_name
    except (ImportError, StopIteration):
        raise FileNotFoundError(
            'Укажите PROTOCOL_URL или положите официальный протокол по пути '
            f'{PROTOCOL_PATH}'
        )

print('Используется файл:', PROTOCOL_PATH)

## Поля официального протокола

Файлы `ASVspoof5.train.tsv`, `ASVspoof5.dev.track_1.tsv` и `ASVspoof5.eval.track_1.tsv` не имеют строки заголовка и содержат 10 полей, разделённых пробелами: `SPEAKER_ID`, `FLAC_FILE_NAME`, `GENDER`, `CODEC`, `CODEC_Q`, `CODEC_SEED`, `ATTACK_TAG`, `ATTACK_LABEL`, `KEY`, `TMP`. Значения класса в `KEY`: `bonafide` и `spoof`. Названия не заменяются придуманной схемой.

In [ ]:
# 4. Чтение протокола через pandas
PROTOCOL_COLUMNS = [
    'SPEAKER_ID', 'FLAC_FILE_NAME', 'GENDER', 'CODEC', 'CODEC_Q',
    'CODEC_SEED', 'ATTACK_TAG', 'ATTACK_LABEL', 'KEY', 'TMP'
]

protocol = pd.read_csv(
    PROTOCOL_PATH,
    sep=r'\s+',
    names=PROTOCOL_COLUMNS,
    header=None,
    dtype='string',
)

if protocol.shape[1] != len(PROTOCOL_COLUMNS):
    raise ValueError('Неожиданное число полей в протоколе')

known_labels = {'bonafide', 'spoof'}
unknown_labels = set(protocol['KEY'].dropna().unique()) - known_labels
if unknown_labels:
    raise ValueError(f'Неизвестные значения KEY: {sorted(unknown_labels)}')

In [ ]:
# 5. Первые строки
protocol.head()

In [ ]:
# 6. Названия столбцов
protocol.columns.tolist()

In [ ]:
# 7. Общее количество записей
print(f'Всего записей: {len(protocol):,}')

In [ ]:
# 8. Количество bonafide/REAL и spoof/FAKE
class_counts = (
    protocol['KEY']
    .value_counts(dropna=False)
    .rename_axis('official_label')
    .reset_index(name='count')
)
class_counts['project_label'] = class_counts['official_label'].map(
    {'bonafide': 'REAL', 'spoof': 'FAKE'}
)
class_counts

In [ ]:
# 9. Распределение FAKE по attack ID
fake_protocol = protocol.loc[protocol['KEY'].eq('spoof')].copy()
attack_counts = (
    fake_protocol['ATTACK_LABEL']
    .value_counts(dropna=False)
    .rename_axis('ATTACK_LABEL')
    .reset_index(name='count')
)
attack_counts

In [ ]:
# 10. Распределения дополнительных полей
useful_fields = ['GENDER', 'CODEC', 'CODEC_Q', 'ATTACK_TAG']
for column in useful_fields:
    print(f'\n{column}')
    display(protocol[column].value_counts(dropna=False).to_frame('count'))

print('\nЧисло уникальных SPEAKER_ID:', protocol['SPEAKER_ID'].nunique())
display(protocol['SPEAKER_ID'].value_counts().head(20).to_frame('count'))

In [ ]:
# 11. Простые графики REAL/FAKE и атак
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.countplot(data=protocol, x='KEY', order=['bonafide', 'spoof'], ax=axes[0])
axes[0].set_title('Распределение REAL/FAKE')
axes[0].set_xlabel('Официальная метка KEY')
axes[0].set_ylabel('Количество')

attack_order = fake_protocol['ATTACK_LABEL'].value_counts().index
sns.countplot(
    data=fake_protocol, y='ATTACK_LABEL', order=attack_order, ax=axes[1]
)
axes[1].set_title('Распределение FAKE по ATTACK_LABEL')
axes[1].set_xlabel('Количество')
axes[1].set_ylabel('Attack ID')

plt.tight_layout()
plt.show()

In [ ]:
# 12. Таблица для решения о первом подмножестве обучения
subset_summary = (
    protocol
    .groupby(['KEY', 'ATTACK_LABEL', 'GENDER', 'CODEC'], dropna=False)
    .size()
    .reset_index(name='record_count')
)
subset_summary['share_within_class_percent'] = (
    subset_summary['record_count']
    / subset_summary.groupby('KEY')['record_count'].transform('sum')
    * 100
).round(2)
subset_summary = subset_summary.sort_values(
    ['KEY', 'record_count'], ascending=[True, False]
)

print(
    'Используйте эту таблицу, чтобы отдельно проверить баланс классов, атак, '
    'пола и кодеков. Конкретное подмножество здесь намеренно не выбирается.'
)
subset_summary